In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


folder = r"C:\Users\sujat\OneDrive\Desktop\Retail-Inv-opt"

sheets = pd.read_excel(
    folder + r"\online_retail_II.xlsx",
    sheet_name=None,
    dtype={"StockCode": str, "Invoice": str},
)
print("Sheets found:", list(sheets.keys()))

df = pd.concat(sheets.values(), ignore_index=True)
df.to_csv(folder + r"\online_retail_II.csv", index=False)

print(df.shape)  # expect (1067371, 8)
df.info()
display(df.head())

In [ ]:
# checking size and structure of the data
print("Shape:", df.shape)
df.info()

In [ ]:
# which columns have nulls
print(df.isna().sum())

In [ ]:
# duplicate rows
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# cancelled invoices start with 'C'
cancelled = df["Invoice"].astype(str).str.startswith("C").sum()
print("Cancelled invoices:", cancelled)

# negative or zero quantity / price
print("Quantity <= 0:", (df["Quantity"] <= 0).sum())
print("Price <= 0:", (df["Price"] <= 0).sum())

In [ ]:
# time period covered
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

# number of unique products and invoices
print("Unique SKUs:", df["StockCode"].nunique())
print("Unique invoices:", df["Invoice"].nunique())

In [ ]:
df.describe()

In [ ]:
# codes like postage, bank charges, manual entries are not real products
non_product = df[df["StockCode"].astype(str).str.match(
    r"^(POST|DOT|M$|BANK CHARGES|C2|AMAZONFEE|ADJUST|S$|B$|CRUK|D$|TEST)", na=False)]

print(non_product["StockCode"].value_counts())

In [ ]:
# work on a copy so raw df stays safe
data = df.copy()
print("Start:", len(data))

data = data.drop_duplicates()
print("After duplicates:", len(data))

In [ ]:
# invoices starting with 'C' are cancelled
data = data[~data["Invoice"].astype(str).str.startswith("C")]
print("After cancellations:", len(data))

In [ ]:
data = data[(data["Quantity"] > 0) & (data["Price"] > 0)]
print("After bad qty/price:", len(data))

In [ ]:
# postage, fees, adjustments etc. are not real products
non_product_codes = ["POST", "DOT", "M", "C2", "D", "S", "BANK CHARGES",
                     "ADJUST", "ADJUST2", "AMAZONFEE", "CRUK", "B",
                     "TEST001", "TEST002", "PADS"]

data = data[~data["StockCode"].isin(non_product_codes)]
print("After non-product codes:", len(data))

In [ ]:
# codes with no digits can still be non-products
odd = data[~data["StockCode"].str.contains(r"\d", regex=True)]
print(odd["StockCode"].value_counts())

# gift vouchers
print("Gift voucher rows:", data["StockCode"].str.lower().str.startswith("gift").sum())

In [ ]:
data["Revenue"] = data["Quantity"] * data["Price"]

# biggest quantities, to spot outliers
data.sort_values("Quantity", ascending=False).head(10)[
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "InvoiceDate"]]

In [ ]:
# two huge orders that were cancelled later (cancel rows already dropped)
data = data[~data["Quantity"].isin([80995, 74215])]
print("After outlier orders:", len(data))

# gift vouchers and lowercase 'm' are not products
data = data[~data["StockCode"].str.lower().str.startswith("gift")]
data = data[data["StockCode"] != "m"]
print("After vouchers and m:", len(data))

In [ ]:
data = data.reset_index(drop=True)
data.to_csv(folder + r"\online_retail_clean.csv", index=False)
print("Clean shape:", data.shape)

In [ ]:
# two huge orders that were cancelled later (cancel rows already dropped)
data = data[~((data["StockCode"] == "23843") & (data["Quantity"] == 80995))]
data = data[~((data["StockCode"] == "23166") & (data["Quantity"] == 74215))]

# gift vouchers and lowercase 'm' are not products
data = data[~data["StockCode"].str.lower().str.startswith("gift")]
data = data[data["StockCode"] != "m"]

data = data.reset_index(drop=True)
data.to_csv(folder + r"\online_retail_clean.csv", index=False)
print("Clean shape:", data.shape)  # expect (1003338, 9)

In [ ]:
# one row per product
sku = data.groupby("StockCode").agg(
    revenue=("Revenue", "sum"),
    units=("Quantity", "sum"),
    orders=("Invoice", "nunique"),
    avg_price=("Price", "mean"),
    first_sale=("InvoiceDate", "min"),
    last_sale=("InvoiceDate", "max"),
).reset_index()

# add description (latest one used for each product)
desc = data.sort_values("InvoiceDate").groupby("StockCode")["Description"].last()
sku["description"] = sku["StockCode"].map(desc)

print("Total SKUs:", len(sku))
sku.head()

In [ ]:
# rank by revenue, then cumulative share
sku = sku.sort_values("revenue", ascending=False).reset_index(drop=True)
sku["rev_share"] = sku["revenue"] / sku["revenue"].sum()
sku["cum_share"] = sku["rev_share"].cumsum()
sku["sku_pct"] = (sku.index + 1) / len(sku)

# A = top 80% revenue, B = next 15%, C = last 5%
def abc(x):
    if x <= 0.80:
        return "A"
    elif x <= 0.95:
        return "B"
    return "C"

sku["ABC"] = sku["cum_share"].apply(abc)

In [ ]:
summary = sku.groupby("ABC").agg(
    sku_count=("StockCode", "count"),
    revenue=("revenue", "sum"),
)
summary["pct_of_skus"] = (summary["sku_count"] / summary["sku_count"].sum() * 100).round(1)
summary["pct_of_revenue"] = (summary["revenue"] / summary["revenue"].sum() * 100).round(1)
summary

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(sku["sku_pct"] * 100, sku["cum_share"] * 100)
plt.axhline(80, color="red", linestyle="--", label="80% revenue")
plt.xlabel("% of SKUs")
plt.ylabel("Cumulative % of revenue")
plt.title("Pareto Curve - Revenue vs SKUs")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# month column
data["month"] = data["InvoiceDate"].dt.to_period("M")

# drop Dec 2011, it is only a partial month
monthly = data[data["month"] < pd.Period("2011-12")]

# units sold per SKU per month (0 where no sales)
demand = monthly.pivot_table(index="StockCode", columns="month",
                             values="Quantity", aggfunc="sum", fill_value=0)
print(demand.shape)  # SKUs x 23 months

In [ ]:
# CV = std / mean of monthly demand (lower = more stable)
cv = demand.std(axis=1) / demand.mean(axis=1)

sku["cv"] = sku["StockCode"].map(cv)

# SKUs with no sales before Dec 2011 have no CV, treat them as erratic
sku["cv"] = sku["cv"].fillna(99)

In [ ]:
# X = stable, Y = moderate, Z = erratic
def xyz(c):
    if c <= 0.5:
        return "X"
    elif c <= 1.0:
        return "Y"
    return "Z"

sku["XYZ"] = sku["cv"].apply(xyz)
sku["ABC_XYZ"] = sku["ABC"] + sku["XYZ"]

# SKU count in each cell
print(pd.crosstab(sku["ABC"], sku["XYZ"], margins=True))

# revenue share (%) in each cell
rev = pd.pivot_table(sku, index="ABC", columns="XYZ", values="revenue", aggfunc="sum")
print((rev / rev.sum().sum() * 100).round(1))

In [ ]:
# CV only between first and last month of sale (ignores months before launch / after discontinuation)
def active_cv(row):
    nz = np.nonzero(row.values)[0]
    window = row.values[nz[0]:nz[-1] + 1]
    if len(window) < 3:          # too short to judge
        return np.nan
    return window.std(ddof=1) / window.mean()

cv_active = demand.apply(active_cv, axis=1)
sku["cv_active"] = sku["StockCode"].map(cv_active)

print("SKUs with life under 3 months:", sku["cv_active"].isna().sum())

# short-life SKUs are treated as erratic
sku["cv_active"] = sku["cv_active"].fillna(99)

In [ ]:
sku["XYZ_adj"] = sku["cv_active"].apply(xyz)

print(pd.crosstab(sku["ABC"], sku["XYZ_adj"], margins=True))

rev = pd.pivot_table(sku, index="ABC", columns="XYZ_adj", values="revenue", aggfunc="sum")
print((rev / rev.sum().sum() * 100).round(1))

In [ ]:
# snapshot = last date in the data
snapshot = data["InvoiceDate"].max()
print("Snapshot date:", snapshot)

# days since each SKU last sold
sku["days_since_sale"] = (snapshot - sku["last_sale"]).dt.days

# Dead = no sale in 180+ days, Slow = 90 to 179 days, Active = sold in last 90 days
def status(d):
    if d >= 180:
        return "Dead"
    elif d >= 90:
        return "Slow"
    return "Active"

sku["status"] = sku["days_since_sale"].apply(status)
print(sku["status"].value_counts())

In [ ]:
print(pd.crosstab(sku["ABC"], sku["status"], margins=True))

In [ ]:
# ASSUMPTIONS - write these in README
cost_ratio = 0.60     # unit cost = 60% of selling price
cover_days = 90       # stock held = 90 days of average demand

# life of each SKU in days (minimum 90, so one-off sales don't give huge daily demand)
life_days = (sku["last_sale"] - sku["first_sale"]).dt.days.clip(lower=90)

sku["daily_demand"] = sku["units"] / life_days
sku["est_stock_units"] = sku["daily_demand"] * cover_days
sku["est_stock_value"] = sku["est_stock_units"] * sku["avg_price"] * cost_ratio

In [ ]:
cap = sku.pivot_table(index="ABC", columns="status", values="est_stock_value",
                      aggfunc="sum", fill_value=0)
cap["Total"] = cap.sum(axis=1)
print(cap.round(0))

total = sku["est_stock_value"].sum()
locked = sku.loc[sku["status"].isin(["Dead", "Slow"]), "est_stock_value"].sum()
print("Total est. stock value:", round(total))
print("Locked in dead + slow:", round(locked), f"({locked / total * 100:.1f}%)")

In [ ]:
# biggest dead A-class items by estimated stock value
dead_a = sku[(sku["ABC"] == "A") & (sku["status"] == "Dead")]
dead_a.sort_values("est_stock_value", ascending=False).head(10)[
    ["StockCode", "description", "revenue", "last_sale", "days_since_sale", "est_stock_value"]]

In [ ]:
# calendar month and year of each sale
data["cal_month"] = data["InvoiceDate"].dt.month
data["cal_year"] = data["InvoiceDate"].dt.year

# units sold per SKU in each calendar month
by_month = data.pivot_table(index="StockCode", columns="cal_month",
                            values="Quantity", aggfunc="sum", fill_value=0)

# share of a SKU's units that falls in its 3 best calendar months
top3_share = by_month.apply(lambda r: r.nlargest(3).sum() / r.sum(), axis=1)
years_sold = data.groupby("StockCode")["cal_year"].nunique()

sku["top3_share"] = sku["StockCode"].map(top3_share)
sku["years_sold"] = sku["StockCode"].map(years_sold)

# seasonal = sales bunched in 3 months AND seen in at least 2 different years
sku["seasonal"] = (sku["top3_share"] >= 0.70) & (sku["years_sold"] >= 2)
print(sku["seasonal"].sum(), "seasonal SKUs")

In [ ]:
flagged = sku[sku["status"].isin(["Dead", "Slow"])]
print(pd.crosstab(flagged["ABC"], flagged["seasonal"], margins=True))

true_locked = flagged.loc[~flagged["seasonal"], "est_stock_value"].sum()
print("Locked excluding seasonal:", round(true_locked),
      f"({true_locked / total * 100:.1f}% of total)")

In [ ]:
for d in [90, 120, 180, 270]:
    v = sku.loc[sku["days_since_sale"] >= d, "est_stock_value"].sum()
    print(f"No sale in {d}+ days: {v / total * 100:.1f}% of est. stock value")

In [ ]:
# seasonal needs at least ~1 year of history and enough orders to judge a pattern
life_full = (sku["last_sale"] - sku["first_sale"]).dt.days
sku["seasonal_strict"] = (sku["top3_share"] >= 0.70) & (life_full >= 330) & (sku["orders"] >= 6)

print(sku["seasonal_strict"].sum(), "seasonal SKUs (strict)")

In [ ]:
flagged = sku[sku["status"].isin(["Dead", "Slow"])]
print(pd.crosstab(flagged["ABC"], flagged["seasonal_strict"], margins=True))

locked_strict = flagged.loc[~flagged["seasonal_strict"], "est_stock_value"].sum()
print("Locked excluding strict seasonal:", round(locked_strict),
      f"({locked_strict / total * 100:.1f}% of total)")

In [ ]:
def strategy(r):
    # dead or slow items
    if r["status"] in ["Dead", "Slow"]:
        if r["seasonal_strict"]:
            return "Seasonal - order to forecast before season"
        if r["ABC"] == "A":
            return "Former top seller - check why it stopped, clear remaining stock"
        return "Delist and liquidate"
    # active items
    if r["ABC"] == "A":
        return {"X": "Tight control, frequent replenishment, low safety stock",
                "Y": "Regular review, moderate safety stock",
                "Z": "Higher safety stock, monitor demand closely"}[r["XYZ_adj"]]
    if r["ABC"] == "B":
        return "Standard reorder point, periodic review" if r["XYZ_adj"] != "Z" \
               else "Small batches, review range"
    return "Bulk and infrequent ordering" if r["XYZ_adj"] != "Z" \
           else "Order on demand, consider cutting from range"

sku["strategy"] = sku.apply(strategy, axis=1)
print(sku["strategy"].value_counts())

In [ ]:
# ASSUMPTIONS - write in README
recovery_rate = 0.50      # cash recovered on liquidated stock
cover_x_new = 60          # new cover days for stable active items

# 1. liquidation of dead/slow non-seasonal stock
liq = sku[sku["strategy"].isin(["Delist and liquidate",
          "Former top seller - check why it stopped, clear remaining stock"])]
freed_liq = liq["est_stock_value"].sum() * recovery_rate

# 2. lower cover on stable active items
stable = sku[(sku["status"] == "Active") & (sku["XYZ_adj"] == "X")]
freed_cover = stable["est_stock_value"].sum() * (1 - cover_x_new / cover_days)

print("Liquidation:", round(freed_liq))
print("Lower cover on X items:", round(freed_cover))
print("Total capital freed:", round(freed_liq + freed_cover),
      f"({(freed_liq + freed_cover) / total * 100:.1f}% of est. stock value)")

In [ ]:
strat_sum = sku.groupby("strategy").agg(
    skus=("StockCode", "count"),
    revenue=("revenue", "sum"),
    est_stock_value=("est_stock_value", "sum"),
).round(0).sort_values("est_stock_value", ascending=False)
strat_sum

In [ ]:
sku.to_csv(folder + r"\sku_analysis.csv", index=False)
print("Saved", sku.shape)

In [ ]:
# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.family': 'sans-serif'})

# ABC-XYZ 3x3 HEATMAP VISUALIZATION
# Pivot tables for SKU counts and Revenue Share %
sku_matrix = pd.pivot_table(sku, index="ABC", columns="XYZ_adj", values="StockCode", aggfunc="count", fill_value=0)
rev_matrix = pd.pivot_table(sku, index="ABC", columns="XYZ_adj", values="revenue", aggfunc="sum", fill_value=0)
rev_share_matrix = (rev_matrix / rev_matrix.sum().sum() * 100).round(1)

# Combined text labels: "Count (Revenue %)"
annot_matrix = sku_matrix.astype(str) + "\n(" + rev_share_matrix.astype(str) + "%)"

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(sku_matrix, annot=annot_matrix, fmt="", cmap="YlGnBu", cbar=False,
            linewidths=1.5, linecolor='white', annot_kws={"size": 11, "weight": "bold"}, ax=ax)

ax.set_title("ABC-XYZ Matrix: SKU Count & (Revenue Share %)", fontsize=14, pad=15, weight='bold')
ax.set_xlabel("XYZ Class (Demand Volatility / CV)", fontsize=12, labelpad=10)
ax.set_ylabel("ABC Class (Revenue Contribution)", fontsize=12, labelpad=10)
plt.tight_layout()
plt.show()


In [ ]:
#  WORKING CAPITAL BREAKDOWN CHART
cap_summary = sku.groupby(["ABC", "status"])["est_stock_value"].sum().unstack(fill_value=0) / 1e3  # In Thousands

fig, ax = plt.subplots(figsize=(9, 5))
cap_summary[["Active", "Slow", "Dead"]].plot(kind="bar", stacked=True, 
                                           color=["#2ca02c", "#ff7f0e", "#d62728"], 
                                           ax=ax, edgecolor="none", width=0.6)

ax.set_title("Estimated Working Capital Tied Up by ABC Class (£ Thousands)", fontsize=14, pad=15, weight='bold')
ax.set_xlabel("ABC Category", fontsize=12, labelpad=10)
ax.set_ylabel("Stock Value (£k)", fontsize=12, labelpad=10)
ax.legend(title="Inventory Status", frameon=True)
plt.xticks(rotation=0)

# Add total values on top of bars
for i, total in enumerate(cap_summary.sum(axis=1)):
    ax.text(i, total + 20, f"${total:,.0f}k", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# SAFETY STOCK & REORDER POINT (ROP)
# Assumptions: lead time = 14 days; service levels A=98% (Z=2.05), B=95% (Z=1.65), C=90% (Z=1.28)
lead_time_days = 14
z_scores = {"A": 2.05, "B": 1.65, "C": 1.28}

# only active SKUs with a real CV (exclude the 99 placeholder)
active_skus = sku[(sku["status"] == "Active") & (sku["cv_active"] < 99)].copy()

active_skus["z_score"] = active_skus["ABC"].map(z_scores)

# monthly demand and its std (CV is monthly, so keep units monthly)
monthly_demand = active_skus["daily_demand"] * 30
monthly_std = monthly_demand * active_skus["cv_active"]

# std over the lead time = monthly std * sqrt(lead time in months)
std_lead_time = monthly_std * np.sqrt(lead_time_days / 30)

# Safety stock = Z * std over lead time
active_skus["safety_stock_units"] = np.ceil(active_skus["z_score"] * std_lead_time)

# ROP = demand during lead time + safety stock
active_skus["rop_units"] = np.ceil(active_skus["daily_demand"] * lead_time_days
                                   + active_skus["safety_stock_units"])

In [ ]:
# Merge ROP back into main SKU dataframe
sku["safety_stock_units"] = sku["StockCode"].map(active_skus.set_index("StockCode")["safety_stock_units"]).fillna(0)
sku["rop_units"] = sku["StockCode"].map(active_skus.set_index("StockCode")["rop_units"]).fillna(0)


In [ ]:
# Save updated dataset
sku.to_csv(folder + r"\sku_analysis_final.csv", index=False)
print("Updated dataframe with ROP and Safety Stock saved successfully!")